In [1]:
%pip install "py4j==0.10.9.7" "pyOpenSSL==25.3.0" "cryptography>=43,<47"
%pip check

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 4.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 8.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.8.0
    Uninstalling typing_extensions-4.8.0:
      Successfully uninstalled typing_extensions-4.8.0
  Attempting uninstall: cffi
    Found existing installation: cffi 1.16.0
    Uninstalling cffi-1.16.0:
      Successfully uninstalled cffi-1.16.0
  Attempting uninstall: cryptography
    Found existing installation: cryptography 41.0.4
    Uninstalling cryptography-41.0.4:
      Successfully uninstalled cryptography-41.0.4
  Attempting uninstall: pyOpenSSL
    Found existing installation:

In [2]:
%pip install "numpy==1.26.4" "numba==0.59.1"
%pip check

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 10.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 9.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 MB 10.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.40.1
    Uninstalling llvmlite-0.40.1:
      Successfully uninstalled llvmlite-0.40.1
  Attempting uninstall: numba
    Found existing installation: numba 0.57.1
    Uninstalling numba-0.57.1:
      Successfully uninstalled numba-0.57.1
Note: you may need to restart the kernel to use updated packages.
No broken requirements found.
Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip -q install mlflow scikit-learn pandas
%pip check

No broken requirements found.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os, pathlib, tempfile
import mlflow
# Backend SQLite local (robuste)
db_path = pathlib.Path(tempfile.gettempdir()) / "mlflow_demo.db"
tracking_uri = f"sqlite:///{db_path}"
mlflow.set_tracking_uri(tracking_uri)
experiment_name = "demo-mlflow-notebook"
mlflow.set_experiment(experiment_name)
# Pour que `!mlflow run` log au même endroit que Python
os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
os.environ["MLFLOW_EXPERIMENT_NAME"] = experiment_name
print("Tracking URI =", tracking_uri)
print("Experiment  =", experiment_name)

2026/09/30 09:13:16 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/30 09:13:16 INFO mlflow.store.db.utils: Updating database tables
2026/09/30 09:13:21 INFO mlflow.tracking.fluent: Experiment with name 'demo-mlflow-notebook' does not exist. Creating a new experiment.


Tracking URI = sqlite:////tmp/mlflow_demo.db
Experiment  = demo-mlflow-notebook


In [5]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.25, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((112, 4), (38, 4))

In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import mlflow
import mlflow.sklearn

params = {
    "n_estimators": 200,
    "max_depth": 10,
    "random_state": 42
}

with mlflow.start_run(run_name="tracking-rf") as run:
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    acc = accuracy_score(y_test, model.predict(X_test))

    mlflow.log_params(params)
    mlflow.log_metric("accuracy", acc)

    fi = pd.Series(
        model.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False)

    fi.to_csv("feature_importances.csv")
    mlflow.log_artifact("feature_importances.csv")

    mlflow.sklearn.log_model(
        model,
        name="model",
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

print("run_id =", run.info.run_id)
print("accuracy =", acc)

run_id = 649cd6b17d524a43983580755b3e9d68
accuracy = 0.8947368421052632


In [7]:
import mlflow
exp = mlflow.get_experiment_by_name(os.environ["MLFLOW_EXPERIMENT_NAME"])
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=["start_time DESC"])
runs[["run_id", "status", "metrics.accuracy"]].head(10)


,run_id,status,metrics.accuracy
0,649cd6b17d524a43983580755b3e9d68,FINISHED,0.894737


In [8]:
import textwrap
import pathlib
import tempfile

project_dir = pathlib.Path(tempfile.gettempdir()) / "mlflow_project_demo_notebook"

# Créer le dossier s’il n’existe pas
project_dir.mkdir(parents=True, exist_ok=True)

# Puis ton code existant
(project_dir / "train.py").write_text(
    textwrap.dedent("""\
import argparse
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier


def main(n_estimators: int, max_depth: int):
    # IMPORTANT (MLflow Projects):
    # - mlflow run crée déjà un run et fournit MLFLOW_RUN_ID / MLFLOW_EXPERIMENT_ID via env
    # - On ne relance pas mlflow.start_run() ici
    iris = load_iris(as_frame=True)
    X = iris.data
    y = iris.target
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    params = {"n_estimators": n_estimators, "max_depth": max_depth, "random_state": 42}
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", acc)
    fi = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
    fi.to_csv("feature_importances.csv")
    mlflow.log_artifact("feature_importances.csv")
    mlflow.sklearn.log_model(model, artifact_path="model")

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--n_estimators", type=int, default=200)
    p.add_argument("--max_depth", type=int, default=5)
    args = p.parse_args()
    main(args.n_estimators, args.max_depth)
"""),
    encoding="utf-8"
)

print("train.py updated:", project_dir / "train.py")




train.py updated: /tmp/mlflow_project_demo_notebook/train.py


In [9]:
import os

os.environ.pop("MLFLOW_EXPERIMENT_NAME", None)
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

project_path = str(project_dir)

!mlflow run "{project_path}" -P n_estimators=300 -P max_depth=4

2026/09/30 09:13:28 ERROR mlflow.cli: === Could not find main among entry points [] or interpret main as a runnable script. Supported script file extensions: ['.py', '.sh'] ===


In [10]:
train_path = project_dir / "train.py"

code = train_path.read_text(encoding="utf-8")

ancien = 'mlflow.sklearn.log_model(model, artifact_path="model")'
nouveau = (
    'mlflow.sklearn.log_model('
    'model, name="model", '
    'skops_trusted_types=["sklearn.tree._tree.Tree"])'
)

if ancien in code:
    train_path.write_text(
        code.replace(ancien, nouveau),
        encoding="utf-8"
    )
    print("train.py corrigé")
else:
    print("Ligne non trouvée. Contenu actuel :")
    print(code)

train.py corrigé


In [11]:
project_path = str(project_dir)

!mlflow run "{project_path}" --env-manager local -P n_estimators=300 -P max_depth=4

2026/09/30 09:13:31 ERROR mlflow.cli: === Could not find main among entry points [] or interpret main as a runnable script. Supported script file extensions: ['.py', '.sh'] ===


In [12]:
import mlflow, mlflow.pyfunc

exp = mlflow.get_experiment_by_name("demo-mlflow-notebook")
last = mlflow.search_runs([exp.experiment_id], order_by=["start_time DESC"], max_results=1)
run_id = last.iloc[0]["run_id"]

mlflow.pyfunc.load_model(f"runs:/{run_id}/model")

mlflow.pyfunc.loaded_model:
  artifact_path: /home/jovyan/mlruns/1/models/m-5078aa32f5d64e7686261932601fb979/artifacts
  flavor: mlflow.sklearn
  run_id: 649cd6b17d524a43983580755b3e9d68

In [13]:
import mlflow, mlflow.pyfunc

exp = mlflow.get_experiment_by_name("demo-mlflow-notebook")
last = mlflow.search_runs([exp.experiment_id], order_by=["start_time DESC"], max_results=1)
run_id = last.iloc[0]["run_id"]

model = mlflow.pyfunc.load_model(f"runs:/{run_id}/model")
print("Loaded model from run_id =", "812439198184cd88a83099e5f61d5c3") #run_id)

Loaded model from run_id = 812439198184cd88a83099e5f61d5c3


In [14]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X = iris.data

pred = model.predict(X.tail(20))
pred

array([2, 2, 2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [15]:
from sklearn.metrics import accuracy_score

y_true = iris.target
y_pred = model.predict(X)

accuracy_score(y_true, y_pred)

0.9733333333333334

In [16]:
print(f'mlflow models serve -m "runs:/{run_id}/model" -p 5001 --no-conda')

mlflow models serve -m "runs:/649cd6b17d524a43983580755b3e9d68/model" -p 5001 --no-conda


In [ ]:
import os
import sys
import mlflow

os.environ["MLFLOW_TRACKING_URI"] = mlflow.get_tracking_uri()

!{sys.executable} -m mlflow models serve -m "runs:/{run_id}/model" --host 0.0.0.0 -p 5001 --env-manager local

2026/09/30 09:13:34 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2026/09/30 09:13:34 INFO mlflow.pyfunc.backend: === Running command 'exec uvicorn --host 0.0.0.0 --port 5001 --workers 1 mlflow.pyfunc.scoring_server.app:app'
INFO:     Started server process [443]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:5001 (Press CTRL+C to quit)
INFO:     172.24.0.1:39282 - "POST /invocations HTTP/1.1" 200 OK
